### Loading The Dataset Online_retail_II.xlsx

In [3]:
import pandas as pd

sheets = pd.read_excel("online_retail_II.xlsx", sheet_name= None)

df_raw = pd.concat(sheets.values(), ignore_index= True)
df_raw.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df = df_raw.copy()

### Data Cleaning

#### Dataset Information

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   Transations  1067371 non-null  object        
 9   LineAmount   1067371 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(5)
memory usage: 81.4+ MB


#### Duplicates

In [24]:
df.duplicated().sum()

np.int64(34335)

In [25]:
df[df.duplicated(keep=False)]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Transations,LineAmount
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Sales,3.75
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Sales,3.75
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,16329.0,United Kingdom,Sales,3.75
367,489517,22319,HAIRCLIPS FORTIES FABRIC ASSORTED,12,2009-12-01 11:34:00,0.65,16329.0,United Kingdom,Sales,7.80
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,16329.0,United Kingdom,Sales,5.10
...,...,...,...,...,...,...,...,...,...,...
1067136,581538,22068,BLACK PIRATE TREASURE CHEST,1,2011-12-09 11:34:00,0.39,14446.0,United Kingdom,Sales,0.39
1067150,581538,23318,BOX OF 6 MINI VINTAGE CRACKERS,1,2011-12-09 11:34:00,2.49,14446.0,United Kingdom,Sales,2.49
1067153,581538,22992,REVOLVER WOODEN RULER,1,2011-12-09 11:34:00,1.95,14446.0,United Kingdom,Sales,1.95
1067160,581538,22694,WICKER STAR,1,2011-12-09 11:34:00,2.10,14446.0,United Kingdom,Sales,2.10


- Total 34335 duplicate rows with exact values in every column.

In [28]:
df = df.drop_duplicates()

In [29]:
df.duplicated().sum()

np.int64(0)

#### Handling Missing values

In [ ]:
df.isnull().sum()


Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
Transations         0
LineAmount          0
dtype: int64

Seems we have missing values in column "Description"- `4275` and "Customer ID" - `235151`
- Description can be filed using column "StockCode"
- We cant drop the CustomerID missing values, as the missing value contributes 23% of all the dataset entries, neither can we fill them cause they will be usefull for :
 `Product sales analysis`
-- `Quantity analysis`
-- `Revenue calculation`
-- `Country analysis`
-- `Invoice-level analysis`
-- `Time-series sales analysis`

##### Filling "Description" column using "StockCode"

In [40]:
description_maping = (df.dropna(subset = ["Description"]).
                      drop_duplicates(subset = ["StockCode"]).
                      set_index("StockCode")["Description"]
                      )

df["Description"] = df["Description"].fillna(df["StockCode"].map(description_maping))

(df["Description"].isna().sum())

np.int64(363)

- There are still 363 missing values in column `Description`.
- That means there are some `StockCodes` that doesnt have any existing Product Description.

In [46]:
missing_desc = df[df["Description"].isna()]
missing = df["Description"].isna()
perc_missing = missing.mean()*100

print(missing_desc[["StockCode", "Description","Customer ID"]].head(20))

print("Total percentage of missing values with `StockCode`,`Description`,`Customer ID` are :", perc_missing)

      StockCode Description  Customer ID
470       21646         NaN          NaN
16186     35983         NaN          NaN
19313     84571         NaN          NaN
20975     35949         NaN          NaN
22854    17013A         NaN          NaN
23542     84841         NaN          NaN
32106    72752C         NaN          NaN
32109    72755B         NaN          NaN
32110    72752D         NaN          NaN
43528    35751A         NaN          NaN
43583    71263G         NaN          NaN
43623     20691         NaN          NaN
43625     20880         NaN          NaN
43626     20859         NaN          NaN
43629     72777         NaN          NaN
43635    79069A         NaN          NaN
43640    84247B         NaN          NaN
43641    84986A         NaN          NaN
43687    35980A         NaN          NaN
43688     84940         NaN          NaN
Total percentage of missing values with `StockCode`,`Description`,`Customer ID` are : 0.0351391432631583


- Above are the `StockCodes` that doesnt have any existing `Description`and they dont have any `Customer ID` either.
- The `363` records with missing descriptions could not be recovered using their `StockCode`, and they also lacked `Customer ID`. Since these records represent approximately `0.03%` of the dataset and provide limited analytical information, they were removed from the cleaned dataset.


In [49]:
df = df.dropna(subset=["Description"])

print(df["Description"].isna().sum())

0


In [50]:
df.isnull().sum()

Invoice             0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
Price               0
Customer ID    234788
Country             0
Transations         0
LineAmount          0
dtype: int64

- Missing Values are handeled. and the Customer_ID hasnt been dropped for further analysis.

#### Handling datatypes

In [51]:
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
Transations            object
LineAmount            float64
dtype: object

- Customer ID need to be changed into object type.

In [54]:
df["Customer ID"] = df["Customer ID"].astype("object")
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID            object
Country                object
Transations            object
LineAmount            float64
dtype: object

### Handing Negatives and Returns

#### Finding negative quantities

In [30]:
negatives = df[df["Quantity"] < 0]
print("Negative Transations :", len(negatives))
print(negatives.head())

Negative Transations : 22496
     Invoice StockCode                    Description  Quantity  \
178  C489449     22087       PAPER BUNTING WHITE LACE       -12   
179  C489449    85206A   CREAM FELT EASTER EGG BASKET        -6   
180  C489449     21895  POTTING SHED SOW 'N' GROW SET        -4   
181  C489449     21896             POTTING SHED TWINE        -6   
182  C489449     22083     PAPER CHAIN KIT RETRO SPOT       -12   

            InvoiceDate  Price  Customer ID    Country            Transations  \
178 2009-12-01 10:33:00   2.95      16321.0  Australia  Returns/Cancellations   
179 2009-12-01 10:33:00   1.65      16321.0  Australia  Returns/Cancellations   
180 2009-12-01 10:33:00   4.25      16321.0  Australia  Returns/Cancellations   
181 2009-12-01 10:33:00   2.10      16321.0  Australia  Returns/Cancellations   
182 2009-12-01 10:33:00   2.95      16321.0  Australia  Returns/Cancellations   

     LineAmount  
178       -35.4  
179        -9.9  
180       -17.0  
181      

#### Finding Return/Cancellations

In [31]:
returns = df[(df["Quantity"]<0) &
             (df["Invoice"].astype(str).str.startswith("C"))
            ]

print("Returns/Cancellations :", len(returns))


Returns/Cancellations : 19103


#### Other Negative Adjustments

In [32]:
others = df[(df["Quantity"]<0) &
             (~df["Invoice"].astype(str).str.startswith("C"))
            ]

print("Negtive Adjustments :", len(others))

Negtive Adjustments : 3393


#### Creating Transaction Classification Column

- Help finding Net Sales, Total sales, Retuned Sales

In [143]:
def transaction_types(row):

    if row["Quantity"] > 0:
        return "Sales"

    elif str(row["Invoice"]).startswith("C"):
        return "Returns/Cancellations"

    else:
        return "Operational Adjustments"

df["Transcations"] = df.apply(transaction_types, axis= 1)
df = df.drop(columns=["Transations"])

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,LineAmount,Transcations
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,Sales
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Sales
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,Sales
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,Sales
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,Sales


#### Calculating Diff Sales Types

In [34]:
df["LineAmount"] = df["Price"]*df["Quantity"]

In [35]:
gross_Sales = df.loc[df["Transations"] == "Sales","LineAmount"].sum()

return_values = df.loc[df["Transations"] == "Returns/Cancellations", "LineAmount"].sum()

Net_sales = df["LineAmount"].sum()

print("Gross_sales = ", gross_Sales)
print("Net Returns = ", return_values)
print("Net Sale = ", Net_sales)

Gross_sales =  20317957.878000002
Net Returns =  -1462424.1800000002
Net Sale =  18855533.698


### Analysis

- Build a revenue and customer activity dashboard showing daily, weekly, and monthly sales trends, with explicit handling of cancellations and returns.  

- Perform cohort analysis based on first purchase month to measure retention, repeat purchase behavior, and long-term revenue contribution.  

- Identify high-value customers using recency, frequency, and monetary value, then analyze product categories driving lifetime value.

#### Cohort Analysis

Steps Include :
- Create a separate cohort-analysis DataFrame
- Exclude transactions with missing Customer ID
- Create PurchaseMonth from InvoiceDate
- Identify each customer's first purchase month
- Assign CohortMonth to every transaction
- Calculate CohortIndex (months since first purchase)
- Calculate monthly active customers by cohort
- Build the cohort retention matrix
- Calculate retention percentages
- Analyze repeat purchase behavior
- Calculate purchase frequency / active months
- Calculate transaction revenue (Quantity × Price)
- Build the cohort revenue matrix
- Analyze cumulative revenue by cohort
- Compare retention across cohorts
- Compare revenue contribution across cohorts
- Identify high-value and low-value cohorts
- Analyze long-term customer value
- Visualize retention and revenue cohorts
- Draw business conclusions and recommendations

In [101]:
cohort_data = df.dropna(subset=["Customer ID"]).copy()

cohort_data["PurchaseMonth"] = (
    cohort_data["InvoiceDate"].dt.to_period("M")
)

customer_first_purchase = (
    cohort_data
    .groupby("Customer ID")["PurchaseMonth"]
    .min()
    .reset_index()
    .rename(columns={"PurchaseMonth": "CohortMonth"})
)

cohort_data = cohort_data.merge(
    customer_first_purchase,
    on="Customer ID",
    how="left"
)

In [102]:
cohort_data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Transations,LineAmount,PurchaseMonth,CohortMonth
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Sales,83.4,2009-12,2009-12
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sales,81.0,2009-12,2009-12
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sales,81.0,2009-12,2009-12
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Sales,100.8,2009-12,2009-12
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Sales,30.0,2009-12,2009-12


##### Calculating Cohort age

In [103]:
cohort_data["CohortIndex"] = (
    (cohort_data["PurchaseMonth"].dt.year -
     cohort_data["CohortMonth"].dt.year) * 12
    +
    (cohort_data["PurchaseMonth"].dt.month -
     cohort_data["CohortMonth"].dt.month)
)

##### Creatting retention matrix

In [104]:
cohort_counts = (
    cohort_data
    .groupby(["CohortMonth", "CohortIndex"])["Customer ID"]
    .nunique()
    .reset_index()
)

In [108]:
retention_table = cohort_counts.pivot(
    index="CohortMonth",
    columns="CohortIndex",
    values="Customer ID"
)
retention_table

CohortIndex,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,1045.0,392.0,358.0,447.0,410.0,408.0,408.0,374.0,355.0,392.0,...,319.0,273.0,316.0,303.0,287.0,274.0,332.0,319.0,427.0,218.0
2010-01,394.0,86.0,119.0,120.0,110.0,115.0,105.0,91.0,114.0,134.0,...,60.0,86.0,74.0,69.0,73.0,93.0,73.0,88.0,29.0,NaN
2010-02,363.0,109.0,82.0,110.0,93.0,76.0,79.0,103.0,100.0,106.0,...,74.0,67.0,61.0,53.0,85.0,90.0,62.0,23.0,NaN,NaN
2010-03,436.0,95.0,113.0,103.0,100.0,87.0,105.0,130.0,126.0,50.0,...,74.0,76.0,69.0,74.0,89.0,93.0,33.0,NaN,NaN,NaN
2010-04,291.0,67.0,58.0,47.0,54.0,67.0,79.0,76.0,33.0,34.0,...,43.0,41.0,41.0,50.0,61.0,19.0,NaN,NaN,NaN,NaN
2010-05,254.0,49.0,45.0,49.0,48.0,66.0,56.0,33.0,17.0,22.0,...,33.0,36.0,42.0,40.0,12.0,NaN,NaN,NaN,NaN,NaN
2010-06,269.0,58.0,53.0,55.0,62.0,76.0,35.0,25.0,22.0,32.0,...,33.0,37.0,55.0,16.0,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,183.0,38.0,37.0,52.0,55.0,28.0,21.0,28.0,26.0,22.0,...,32.0,45.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,158.0,39.0,50.0,51.0,29.0,21.0,16.0,22.0,23.0,21.0,...,32.0,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
retention_pct = retention_table.div(
    retention_table.iloc[:, 0],
    axis=0
) * 100

In [115]:
retention_pct

CohortIndex,0,1,2,3,4,5,6,7,8,9,...,15,16,17,18,19,20,21,22,23,24
CohortMonth,,,,,,,,,,,,,,,,,,,,,
2009-12,100.0,37.511962,34.258373,42.775120,39.234450,39.043062,39.043062,35.789474,33.971292,37.511962,...,30.526316,26.124402,30.239234,28.995215,27.464115,26.220096,31.770335,30.526316,40.861244,20.861244
2010-01,100.0,21.827411,30.203046,30.456853,27.918782,29.187817,26.649746,23.096447,28.934010,34.010152,...,15.228426,21.827411,18.781726,17.512690,18.527919,23.604061,18.527919,22.335025,7.360406,NaN
2010-02,100.0,30.027548,22.589532,30.303030,25.619835,20.936639,21.763085,28.374656,27.548209,29.201102,...,20.385675,18.457300,16.804408,14.600551,23.415978,24.793388,17.079890,6.336088,NaN,NaN
2010-03,100.0,21.788991,25.917431,23.623853,22.935780,19.954128,24.082569,29.816514,28.899083,11.467890,...,16.972477,17.431193,15.825688,16.972477,20.412844,21.330275,7.568807,NaN,NaN,NaN
2010-04,100.0,23.024055,19.931271,16.151203,18.556701,23.024055,27.147766,26.116838,11.340206,11.683849,...,14.776632,14.089347,14.089347,17.182131,20.962199,6.529210,NaN,NaN,NaN,NaN
2010-05,100.0,19.291339,17.716535,19.291339,18.897638,25.984252,22.047244,12.992126,6.692913,8.661417,...,12.992126,14.173228,16.535433,15.748031,4.724409,NaN,NaN,NaN,NaN,NaN
2010-06,100.0,21.561338,19.702602,20.446097,23.048327,28.252788,13.011152,9.293680,8.178439,11.895911,...,12.267658,13.754647,20.446097,5.947955,NaN,NaN,NaN,NaN,NaN,NaN
2010-07,100.0,20.765027,20.218579,28.415301,30.054645,15.300546,11.475410,15.300546,14.207650,12.021858,...,17.486339,24.590164,9.289617,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2010-08,100.0,24.683544,31.645570,32.278481,18.354430,13.291139,10.126582,13.924051,14.556962,13.291139,...,20.253165,6.962025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### customer-level RFM Analysis

Cohort analysis asks:  

`How does a group of customers behave over time?`  

RFM analysis asks:  

`Which individual customers are currently most valuable, and what do those customers buy?`  

Together, these two analyses give you a much stronger picture of retention + customer value + product drivers.

**Business insights this analysis can produce**  

Once you've calculated the actual numbers, the important findings will be things like:  

- High-value customer concentration  
- What percentage of customers are classified as high value?  
- What percentage of total revenue do they generate?  
- Is revenue heavily concentrated among a small customer group?  
- Customer behavior  
- Do high-value customers purchase more frequently?  
- How much more recently do they purchase compared with other customers?  
- How large is the monetary difference between high-value and low-value customers?  
- Product drivers  
- Which products generate the most revenue among high-value customers?  
- Which categories have the highest share of high-value customers?  
- Are certain products strongly associated with repeat purchasing?  
- Business opportunity  
- Which products could be used for cross-selling?  
- Which high-value customers appear to be becoming inactive?  
- Which product categories are associated with long-term customer value?  

**RFM Analysis — Analytical Flow**

```text
Transaction Data
        ↓
Customer-Level Aggregation
        ↓
Recency + Frequency + Monetary Value
        ↓
RFM Scoring
        ↓
Customer Segmentation
        ↓
Identify High-Value Customers
        ↓
Connect High-Value Customers back to Transaction Data
        ↓
Product / Category Analysis
        ↓
Identify Products & Categories driving Customer Value
        ↓
Compare Revenue Contribution
        ↓
Identify Retention & Cross-Sell Opportunities
        ↓
Business Recommendations
```


#### Creating RFM Table

In [117]:
analysis_date = cohort_data["InvoiceDate"].max()

rfm = cohort_data.groupby("Customer ID").agg(
    Recency=(
        "InvoiceDate",
        lambda x: (analysis_date - x.max()).days
    ),

    Frequency = (
        "InvoiceDate",
        "nunique"
    ),

    Monetary = (
        "LineAmount",
        "sum"
    )
).reset_index()

In [119]:
rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,325,17,-51.74
1,12347.0,1,8,4921.53
2,12348.0,74,5,2019.40
3,12349.0,18,5,4404.54
4,12350.0,309,1,334.40


#### RFM score calculation

In [127]:
rfm["R-Score"] = pd.qcut(rfm["Recency"], 
                         5, 
                         labels= [5,4,3,2,1]
                         )
rfm["F-Score"] = pd.qcut(rfm["Frequency"].rank(method="first"),
                         5,
                         labels=[1,2,3,4,5],
                         duplicates="drop"
                         )
rfm["M-Score"] = pd.qcut(rfm["Monetary"],
                         5,
                         labels= [1,2,3,4,5]
                         )

rfm["RFM_Score"] = (
    rfm["R-Score"].astype(int)
    + rfm["F-Score"].astype(int)
    + rfm["M-Score"].astype(int)
)

In [128]:
rfm

,Customer ID,Recency,Frequency,Monetary,R-Score,F-Score,M-Score,RFM_Score
0,12346.0,325,17,-51.74,2,5,1,8
1,12347.0,1,8,4921.53,5,4,5,14
2,12348.0,74,5,2019.40,3,3,4,10
3,12349.0,18,5,4404.54,4,3,5,12
4,12350.0,309,1,334.40,2,1,2,5
...,...,...,...,...,...,...,...,...
5937,18283.0,3,22,2664.90,5,5,4,14
5938,18284.0,429,2,436.68,1,2,2,5
5939,18285.0,660,1,427.00,1,2,2,5
5940,18286.0,476,3,1188.43,1,3,4,8


- RFM table created
- R,F,M scores assigned.
- Total RFM score also calculated.
- Now next step will be finding valuable cutomers.

#### Creating customer segments

In [129]:
rfm["RFM_Score"].value_counts().sort_index()

RFM_Score
3     350
4     472
5     476
6     503
7     455
8     493
9     511
10    453
11    495
12    434
13    427
14    391
15    482
Name: count, dtype: int64

In [130]:
def segment_customer(row):
    score = row["RFM_Score"]

    if score >= 13:
        return "High Value"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    else:
        return "At Risk"

rfm["CustomerSegment"] = rfm.apply(
    segment_customer,
    axis=1
)

In [131]:
rfm

,Customer ID,Recency,Frequency,Monetary,R-Score,F-Score,M-Score,RFM_Score,CustomerSegment
0,12346.0,325,17,-51.74,2,5,1,8,Potential
1,12347.0,1,8,4921.53,5,4,5,14,High Value
2,12348.0,74,5,2019.40,3,3,4,10,Loyal
3,12349.0,18,5,4404.54,4,3,5,12,Loyal
4,12350.0,309,1,334.40,2,1,2,5,At Risk
...,...,...,...,...,...,...,...,...,...
5937,18283.0,3,22,2664.90,5,5,4,14,High Value
5938,18284.0,429,2,436.68,1,2,2,5,At Risk
5939,18285.0,660,1,427.00,1,2,2,5,At Risk
5940,18286.0,476,3,1188.43,1,3,4,8,Potential


In [132]:
rfm.groupby("CustomerSegment").agg(
    Customers=("Customer ID", "nunique"),
    AvgRecency=("Recency", "mean"),
    AvgFrequency=("Frequency", "mean"),
    AvgMonetary=("Monetary", "mean"),
    TotalRevenue=("Monetary", "sum")
)

,Customers,AvgRecency,AvgFrequency,AvgMonetary,TotalRevenue
CustomerSegment,,,,,
At Risk,1801,415.067185,1.388118,219.820790,3.958972e+05
High Value,1300,23.630000,21.881538,9291.494076,1.207894e+07
Loyal,1382,94.106368,6.571635,1951.076284,2.696387e+06
Potential,1459,199.747087,3.156957,766.802140,1.118764e+06


#### Mapping CustomerSegment into main datafraame

In [ ]:
customer_segment = rfm[["Customer ID","CustomerSegment"]]

cohort_data = cohort_data.merge(
    customer_segment,
    on="Customer ID",
    how= "left"
)

In [138]:
cohort_data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Transations,LineAmount,PurchaseMonth,CohortMonth,CohortIndex,CustomerSegment
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,Sales,83.4,2009-12,2009-12,0,Loyal
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sales,81.0,2009-12,2009-12,0,Loyal
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,Sales,81.0,2009-12,2009-12,0,Loyal
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,Sales,100.8,2009-12,2009-12,0,Loyal
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,Sales,30.0,2009-12,2009-12,0,Loyal


#### Products driving high-value customer revenue

In [ ]:
high_value_products = cohort_data[cohort_data["CustomerSegment"] == "High Value"].groupby(
    "Description").agg(
        Revenue = ("LineAmount","sum"),
        units = ("Quantity","sum"),
        Customers = ("Customer ID","nunique")
    )

In [141]:
high_value_products

,Revenue,units,Customers
Description,,,
DOORMAT UNION JACK GUNS AND ROSES,620.25,99,25
3 STRIPEY MICE FELTCRAFT,855.00,464,54
4 PURPLE FLOCK DINNER CANDLES,342.88,232,21
50'S CHRISTMAS GIFT BAG LARGE,1631.00,1372,64
ANIMAL STICKERS,45.57,217,6
...,...,...,...
ZINC T-LIGHT HOLDER STARS SMALL,2743.47,3477,107
ZINC TOP 2 DOOR WOODEN SHELF,838.90,142,29
ZINC WILLIE WINKIE CANDLE STICK,3542.82,4302,141
